<a href="https://colab.research.google.com/github/chiragpatel11226/machinelearningjournetpart1/blob/main/eda.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!wget https://raw.githubusercontent.com/chiragpatel11226/machinelearningjourtpart1/main/insurance.csv
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')



In [2]:
df=pd.read_csv('insurance.csv')

FileNotFoundError: [Errno 2] No such file or directory: 'insurance.csv'

In [ ]:
df

In [ ]:
df.head()

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
df.isnull().sum()

In [ ]:
numeric_columns= ['age',  'bmi', 'children', 'charges']
for col in numeric_columns:
    plt.figure(figsize=(6,4))
    sns.distplot(df[col])
    plt.show()

In [ ]:
df.columns

In [ ]:
sns.countplot(x=df['children'])

In [ ]:
sns.countplot(x=df['sex'])

In [ ]:
sns.countplot(x=df['smoker'])

In [ ]:
for col in numeric_columns:
    plt.figure(figsize=(6,4))
    sns.boxplot(x=df[col])
    plt.show()

In [ ]:
plt.figure(figsize=(8,6))
sns.heatmap(df[numeric_columns].corr(),annot=True)
plt.show()

data cleaning and preprocessing


In [ ]:
df_cleaned=df.copy()

In [ ]:
df_cleaned.head()

In [ ]:
df_cleaned.drop_duplicates(inplace= True)

In [ ]:
df_cleaned['sex'].value_counts()

In [ ]:
df_cleaned['sex'] = df_cleaned['sex'].map({'male':1,'female':0})

In [ ]:
df_cleaned.head()

In [ ]:
df_cleaned['smoker']=df_cleaned['smoker'].map({'yes':1,'no':0})

In [ ]:
df_cleaned.head()

In [ ]:
df_cleaned['region'].value_counts()

In [ ]:
df_cleaned.rename(columns={'sex':'is_male',  'smoker':'is_smoker'},inplace=True)

In [ ]:
df_cleaned.head()

In [ ]:
df_cleaned=pd.get_dummies(df_cleaned,columns=['region'])

In [ ]:
df_cleaned.head()

In [ ]:
df_cleaned=df_cleaned.astype(int)

In [ ]:
df_cleaned.head()

feature engineering and extraction

In [ ]:
sns.histplot(df['bmi'])

In [ ]:
df_cleaned['bmi_category']=pd.cut(df_cleaned['bmi'],bins=[0,18.5,24.9,29.9,100],labels=['underweight','healthy','overweight','obese'])

In [ ]:
df_cleaned.head()


In [ ]:
df_cleaned=pd.get_dummies(df_cleaned,columns=['bmi_category'])

In [ ]:
df_cleaned.head()

In [ ]:
df_cleaned=df_cleaned.astype(int)

In [ ]:
df_cleaned.head()

In [ ]:
from sklearn.preprocessing import StandardScaler
cols=['age','bmi','children']
scaler=StandardScaler()
df_cleaned[cols]=scaler.fit_transform(df_cleaned[cols])
df_cleaned

In [ ]:
from scipy.stats import pearsonr
selected_columns=['age','bmi','children','is_male','is_smoker','region_northeast','region_northwest','region_southeast','region_southwest','bmi_category_healthy','bmi_category_obese','bmi_category_overweight','bmi_category_obese']

correlation= {
     feature: pearsonr(df_cleaned[feature],df_cleaned['charges'])[0]
     for feature in selected_columns
}
correlation_df=pd.DataFrame(list(correlation.items()),columns=['feature','correlation'])
correlation_df.sort_values(by='correlation',ascending=False,inplace=True)



In [ ]:
correlation_df

In [ ]:
cat_features=['is_male','is_smoker','region_northeast',
              'region_northwest','region_southeast','region_southwest',
              'bmi_category_healthy','bmi_category_obese','bmi_category_overweight','bmi_category_obese']

In [ ]:
from scipy.stats import chi2_contingency
import pandas as pd
alpha=0.05
df_cleaned["charge_binned"] = pd.qcut(df_cleaned["charges"], q=4, labels=False)
chi2_results = {}
for col in cat_features:
    contingency_table = pd.crosstab(df_cleaned[col], df_cleaned["charge_binned"])
    chi2_stat, p_value, _, _ = chi2_contingency(contingency_table)
    decision ='reject null (keep feature )' if p_value < alpha else 'accept null (drop feature)'
    chi2_results[col] = {"chi2_stat": chi2_stat, "p_value": p_value, "decision": decision}
chi2_results_df = pd.DataFrame(chi2_results).T
chi2_results_df = chi2_results_df.sort_values(by="p_value")
print(chi2_results_df)

In [ ]:
final_df=df_cleaned.drop(['region_northeast','region_northwest','region_southwest','bmi_category_healthy','bmi_category_obese','bmi_category_overweight'],axis=1)
final_df